# Labelling NISAR chips — Colab walkthrough

**Read this first.** Colab cannot run QGIS. QGIS is a desktop program and Colab is a
remote Linux machine with no screen. So the work splits in two:

| Where | What happens |
|---|---|
| **Colab** (this notebook) | cut your chips, create your annotation files, export the final masks |
| **Your laptop** (QGIS) | draw the polygons |

You will run cells 1–6 here, download a small zip, draw in QGIS, upload the result back,
then run cells 8–10. The whole round trip is about 10 minutes of computer time; the
drawing is the part that actually takes a while.

Everything is driven by the `nisar-flood` command line, so nothing in this notebook is
special — you can run the identical commands on your own machine if you prefer.


## 1. Install the project

Takes about two minutes. Run it once per Colab session — Colab forgets everything when
the runtime disconnects.


In [ ]:
!pip install -q rasterio h5py geopandas
!git clone -q --branch feat/28-labelling-tooling \
    https://github.com/cs59300-deep-learning/nisar-flood-mapping.git
%cd nisar-flood-mapping
!pip install -q -e .

import subprocess
print(subprocess.run(['nisar-flood','--help'], capture_output=True, text=True).stdout)


## 2. Say who you are and which chips are yours

`MY_CHIPS` are grid positions inside the granule, written as `row,col`. Varun assigns
these so no two people label the same chip by accident — except in the calibration round
(#29), where everyone is given the *same* list on purpose.

A chip's position becomes its name: position `8800,13200` produces
`N023_20260618_008800_013200.tif`. That is why two people running this notebook with the
same position get byte-identical chips, and why a filename tells you exactly where in
the granule it came from.


In [ ]:
LABELLER = 'your-github-handle'      # <-- change this

GRANULE_ID = 'N023_20260618'          # short name used as the filename prefix
ACQUIRED   = '2026-06-18'             # acquisition date of the granule

MY_CHIPS = [
    '8800,13200',
    '7600,5600',
]

assert LABELLER != 'your-github-handle', 'put your own handle in LABELLER first'
print(f'{LABELLER} is labelling {len(MY_CHIPS)} chips from {GRANULE_ID}')


## 3. Get the granule

The granule is a 1.8 GB `.h5` file. **Option A is the normal one** — Varun puts the file
in the shared Drive folder once and everybody reads it from there, instead of four people
each downloading 1.8 GB from NASA.

Use Option B only if you are working outside the team's Drive.


In [ ]:
# --- Option A: shared Google Drive (recommended) ---
from google.colab import drive
drive.mount('/content/drive')

GRANULE = ('/content/drive/MyDrive/nisar-flood-mapping/raw/'
           'NISAR_L2_PR_GCOV_023_068_D_090_2005_DHDH_A_'
           '20260618T222452_20260618T222527_P05023_N_F_J_001.h5')

import os
print('found' if os.path.exists(GRANULE) else 'NOT FOUND - check the path', GRANULE)


In [ ]:
# --- Option B: download from NASA Earthdata yourself ---
# Needs a free Earthdata account: https://urs.earthdata.nasa.gov/users/new
# You type your own password; it is never stored in the notebook or the repo.
#
# !pip install -q earthaccess
# import earthaccess
# earthaccess.login()                       # prompts you, writes ~/.netrc in this session
# results = earthaccess.search_data(short_name='NISAR_L2_GCOV_V1',
#                                   granule_name='*023_068_D_090*DHDH*20260618*')
# GRANULE = earthaccess.download(results[:1], 'data/raw')[0]
# print(GRANULE)


## 4. Cut your chips

`--only` is what makes this cheap. Without it the command converts the whole granule
(about 5 minutes and 1.5 GB); with it, only your windows are read out of the HDF5, which
takes a few seconds each.

A chip is 512×512 pixels at 20 m, so 10.24 km across. Two bands: HH and HV, both
gamma-nought in decibels. Any position where fewer than 70% of pixels are usable is
skipped automatically, so do not worry if you get back fewer chips than you asked for.


In [ ]:
# IPython substitutes {expression} into a ! command; precomputing the string
# keeps the quoting simple.
ONLY = ' '.join(MY_CHIPS)

!nisar-flood make-chips \
    --granule "{GRANULE}" \
    --granule-id {GRANULE_ID} \
    --date {ACQUIRED} \
    --out data/chips \
    --only {ONLY}


## 5. Create your annotation files

This writes one `.geojson` per chip, named to match it. Each starts with a single
placeholder polygon whose `class` is empty — that exists only so QGIS recognises the file
as a polygon layer and lets you draw into it. The export ignores it. Do not delete it,
and do not give it a class.

The command refuses to overwrite an annotation file that already exists, so running this
twice can never destroy work you have already done.


In [ ]:
!nisar-flood label-init \
    --chips data/chips \
    --out data/labels/annotations \
    --labeller {LABELLER}

!ls -la data/labels/annotations/


## 6. Look at what you are about to label

Worth ten seconds before you start drawing. Water is dark in both bands; forest is bright
and speckled. The `-25 to +5 dB` stretch below is the same one you will set in QGIS, so
what you see here is what you will see there.


In [ ]:
import matplotlib.pyplot as plt, rasterio, glob

paths = sorted(glob.glob('data/chips/*.tif'))
fig, axes = plt.subplots(len(paths), 2, figsize=(9, 4.6*len(paths)), squeeze=False)
for row, path in enumerate(paths):
    with rasterio.open(path) as chip:
        for col, band in enumerate((1, 2)):
            axes[row][col].imshow(chip.read(band), cmap='gray', vmin=-25, vmax=5)
            axes[row][col].set_title(f"{chip.descriptions[band-1]}\n{path.split('/')[-1]}",
                                     fontsize=8)
            axes[row][col].axis('off')
plt.tight_layout(); plt.show()


## 7. Download the pair, then draw in QGIS

Run the cell to get a zip, then do the drawing **on your laptop**.


In [ ]:
import shutil
from google.colab import files

shutil.make_archive(f'/content/{LABELLER}_to_label', 'zip', 'data')
files.download(f'/content/{LABELLER}_to_label.zip')


### In QGIS

Unzip it. You have a `chips/` folder and a `labels/annotations/` folder.

1. **Install QGIS** if you have not: <https://qgis.org/download/>. Free, all platforms.
2. **Drag the `.tif` in.** Two bands appear: 1 is HH, 2 is HV.
3. **Make water visible.** Layer Properties → Symbology → *Singleband gray*, band 1,
   Min `-25`, Max `5`. Water goes near-black, forest mid-grey. Switch to band 2 (HV) to
   cross-check anything that looks ambiguous — water is dark in both.
4. **Drag the matching `.geojson` in.** Same name as the chip.
5. **Click the pencil** (Toggle Editing), then **Add Polygon Feature**. Draw around a
   region, right-click to finish. A form pops up.
6. **Fill the form:**

   | Field | What to put |
   |---|---|
   | `class` | `1` water, `2` suspected flooded vegetation, `-1` unusable/unsure |
   | `labeller` | your GitHub handle |
   | `labelled_on` | today's date, `YYYY-MM-DD` |
   | `evidence_tier` | `1` backscatter, `2` HAND+gauge, `3` reference layer |
   | `notes` | why — required for anything you marked `2` or `-1` |

7. **Save** (Ctrl+S), then click the pencil again to stop editing.

**You never draw class `0`.** Anything you leave undrawn becomes non-water automatically.
That is most of the chip, and drawing it would be hours of wasted work.

**Read `docs/labelling_protocol.md` before you decide anything.** The short version: dark
and low-lying is water; vegetation-textured but low-lying during high water is `2`, not
water; dark but high ground is radar shadow, which is `-1`, not water. When two labelled
classes both seem defensible, use `-1` — an abstained pixel costs nothing, a wrong one
corrupts the IoU.

**Gauge context for this granule:** Manaus stage was **28.47 m on 18 Jun 2026**, the 88th
percentile of the year and 12 days before the annual peak. The floodplain *is* flooded, so
expect a lot of class `2`.


## 8. Upload your finished annotations

Upload the `.geojson` files you edited. Nothing else — the chips are already here.


In [ ]:
from google.colab import files
import shutil, pathlib

pathlib.Path('data/labels/annotations').mkdir(parents=True, exist_ok=True)
for name in files.upload():
    shutil.move(name, f'data/labels/annotations/{name}')
    print('placed', name)


## 9. Export to label masks

This burns your polygons onto the chip's exact pixel grid. The rules it applies, in order:
water, then excluded, then no-data — so where two polygons overlap, the more cautious
label wins. Pixels outside the radar swath are forced to `-1` no matter what you drew over
them.

The output is an `int16` GeoTIFF per chip plus a `labels_manifest.csv` counting every
class, which is what goes in the report.


In [ ]:
!nisar-flood label-export \
    --chips data/chips \
    --annotations data/labels/annotations \
    --out data/labels/masks


## 10. Check it before you hand it in

Three things worth confirming: the mask lines up with the chip, the class mix is
believable, and you have not accidentally labelled the whole chip as water.


In [ ]:
import numpy as np, rasterio, glob

NAMES = {1: 'water', 0: 'non-water', 2: 'excluded', -1: 'no-data'}
for mask_path in sorted(glob.glob('data/labels/masks/*_label.tif')):
    chip_path = mask_path.replace('/labels/masks/', '/chips/').replace('_label.tif', '.tif')
    with rasterio.open(mask_path) as mask, rasterio.open(chip_path) as chip:
        labels = mask.read(1)
        aligned = mask.transform == chip.transform and mask.crs == chip.crs
    print(f"\n{mask_path.split('/')[-1]}   grid matches chip: {aligned}")
    for value, count in zip(*np.unique(labels, return_counts=True)):
        print(f'   {value:>3}  {NAMES.get(value,"?"):12s} {count:7d} px  {count/labels.size:6.1%}')


In [ ]:
# Side by side: what you labelled, over what you labelled it on.
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

cmap = ListedColormap(['#bdbdbd', '#f4f4f4', '#1f6fb4', '#e08a1e'])   # -1, 0, 1, 2
norm = BoundaryNorm([-1.5, -0.5, 0.5, 1.5, 2.5], cmap.N)
for mask_path in sorted(glob.glob('data/labels/masks/*_label.tif')):
    chip_path = mask_path.replace('/labels/masks/', '/chips/').replace('_label.tif', '.tif')
    with rasterio.open(chip_path) as chip, rasterio.open(mask_path) as mask:
        fig, ax = plt.subplots(1, 2, figsize=(9, 4.6))
        ax[0].imshow(chip.read(1), cmap='gray', vmin=-25, vmax=5); ax[0].set_title('HH dB')
        ax[1].imshow(mask.read(1), cmap=cmap, norm=norm); ax[1].set_title('your labels')
        for a in ax: a.axis('off')
        fig.suptitle(mask_path.split('/')[-1], fontsize=9)
        plt.tight_layout(); plt.show()


## 11. Hand it back

Only the annotations and the manifest go into git — the chips and masks are too big and
are gitignored on purpose. Download this zip and attach it to your issue, or commit the
`.geojson` files directly if you have push access.

Then tick your box on the issue and say which evidence tier you leaned on. That note is
what makes the calibration round (#29) useful instead of just a number.


In [ ]:
import shutil
from google.colab import files

shutil.make_archive(f'/content/{LABELLER}_labelled', 'zip', 'data/labels')
files.download(f'/content/{LABELLER}_labelled.zip')


---
## Appendix — how chip positions get chosen

For Varun, not for labellers. Picks positions on the stride-400 grid by how much open
water they contain, so the set has a usable mix instead of 50 chips of plain forest.
A chip that is 95% water teaches nothing; one that is 0% water teaches less.


In [ ]:
import h5py, numpy as np

SIZE, STRIDE, STEP = 512, 400, 4     # STEP decimates the scan to keep it quick

grid = h5py.File(GRANULE, 'r')['science/LSAR/GCOV/grids/frequencyA']
height, width = grid['HHHH'].shape
small, mask = grid['HHHH'][::STEP, ::STEP], grid['mask'][::STEP, ::STEP]
with np.errstate(divide='ignore', invalid='ignore'):
    decibels = 10 * np.log10(small)
decibels = np.where(np.isfinite(decibels), decibels, np.nan)

valid = mask == 1
water = valid & (decibels < -15)      # open water is specular: very dark

def integral(image):
    out = np.zeros((image.shape[0]+1, image.shape[1]+1), np.int64)
    out[1:, 1:] = image.astype(np.int64).cumsum(0).cumsum(1)
    return out

water_sum, valid_sum = integral(water), integral(valid)
side = SIZE // STEP
def total(table, r, c):
    return table[r+side, c+side] - table[r, c+side] - table[r+side, c] + table[r, c]

def starts(length):
    out = list(range(0, length - SIZE + 1, STRIDE))
    if out[-1] != length - SIZE: out.append(length - SIZE)
    return out

candidates = []
for row in starts(height):
    for col in starts(width):
        r, c = row // STEP, col // STEP
        if r + side > water.shape[0] or c + side > water.shape[1]: continue
        if total(valid_sum, r, c) / side**2 < 0.999: continue
        candidates.append((total(water_sum, r, c) / side**2, row, col))

mixed = sorted((c for c in candidates if 0.25 <= c[0] <= 0.50),
               key=lambda c: abs(c[0] - 0.35))
print(f'{len(candidates)} fully-valid positions, {len(mixed)} with a 25-50% water mix\n')
for fraction, row, col in mixed[:12]:
    print(f"    '{row},{col}',    # water ~{fraction:.0%}")
